<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Image%20Caption%20Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Image Caption Generator (CNN + LSTM)

This notebook will guide you through building an Image Caption Generator using a Convolutional Neural Network (CNN) for feature extraction and a Long Short-Term Memory (LSTM) network for sequence generation.

### 1. Setup and Imports

First, let's install the necessary libraries and import them.

In [1]:
# Install necessary libraries
!pip install tensorflow keras numpy pandas matplotlib scikit-image nltk tqdm

In [2]:
# Import essential libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.utils import to_categorical
from tqdm.notebook import tqdm
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### 2. Data Loading and Preprocessing

For image captioning, we typically use datasets like MS COCO or Flickr8k/Flickr30k. For this example, we'll assume a dataset with images and corresponding captions is available. If not, you'll need to download one (e.g., Flickr8k).

Let's define paths and load the caption data.

In [3]:
# Download the Flickr8k dataset (if not already present)
# The dataset consists of images and a text file containing captions.
# You might need to adjust the paths based on where you download/upload your data.

# For demonstration, we'll use a public link, but for persistent use,
# you'd typically upload to Colab or Google Drive.

# This link is for a smaller version or a common one found online.
# For a complete dataset, please refer to official sources or common Kaggle datasets.

!wget -q -O flickr8k.zip "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip"
!unzip -q flickr8k.zip -d Flickr8k_Dataset

!wget -q -O flickr8k_text.zip "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip"
!unzip -q flickr8k_text.zip -d Flickr8k_text

print("Flickr8k dataset downloaded and extracted.")

# Define paths
IMAGES_DIR = 'Flickr8k_Dataset/Flicker8k_Dataset/'
CAPTIONS_FILE = 'Flickr8k_text/Flickr8k.token.txt'

# Load the captions
def load_descriptions(filename):
    file = open(filename, 'r')
    text = file.read()
    file.close()
    descriptions = dict()
    for line in text.split('\n'):
        if line == '':
            continue
        try:
            image_id, image_caption = line.split('\t')
        except ValueError:
            # Handle cases where a line might not split correctly (e.g., blank lines, malformed)
            continue
        image_id = image_id.split('#')[0]
        if image_id not in descriptions:
            descriptions[image_id] = list()
        descriptions[image_id].append(image_caption)
    return descriptions

descriptions = load_descriptions(CAPTIONS_FILE)
print(f'Loaded {len(descriptions)} descriptions.')

# Clean the descriptions
import string

def clean_descriptions(descriptions):
    # Prepare translation table for removing punctuation
    table = str.maketrans('', '', string.punctuation)
    for key, desc_list in descriptions.items():
        for i in range(len(desc_list)):
            desc = desc_list[i]
            # tokenize
            desc = desc.split()
            # convert to lower case
            desc = [word.lower() for word in desc]
            # remove punctuation from each token
            desc = [w.translate(table) for w in desc]
            # remove hanging 's' and a
            desc = [word for word in desc if len(word) > 1]
            # remove tokens with numbers in them
            desc = [word for word in desc if word.isalpha()]
            # store as string
            desc_list[i] =  'startseq ' + ' '.join(desc) + ' endseq'

clean_descriptions(descriptions)

# Build vocabulary
vocabulary = set()
for key in descriptions.keys():
    for d in descriptions[key]:
        [vocabulary.add(word) for word in d.split()]
print(f'Vocabulary Size: {len(vocabulary)}')

# Convert descriptions to a list for easier processing
all_captions = []
for key, desc_list in descriptions.items():
    for desc in desc_list:
        all_captions.append(desc)

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1

print(f'Tokenizer vocabulary size: {vocab_size}')

# Determine the maximum sequence length
max_length = max(len(s.split()) for s in all_captions)
print(f'Max Description Length: {max_length}')

Flickr8k dataset downloaded and extracted.
Loaded 8092 descriptions.
Vocabulary Size: 8765
Tokenizer vocabulary size: 8766
Max Description Length: 34


### 3. Feature Extraction from Images

We will use a pre-trained Convolutional Neural Network (CNN) to extract features from each image. These features will then be used as input to our LSTM model.

In [4]:
# Load a pre-trained CNN model (e.g., ResNet50) for feature extraction
# We will remove the top (classification) layer
feature_extractor = ResNet50(weights='imagenet', include_top=False, pooling='avg')

# Function to extract features from a single image
def extract_features(image_path):
    image = load_img(image_path, target_size=(224, 224))
    image = img_to_array(image)
    image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    image = tf.keras.applications.resnet50.preprocess_input(image)
    feature = feature_extractor.predict(image, verbose=0)
    return feature

# Extract features for all images
features = {}
for img_name in tqdm(os.listdir(IMAGES_DIR)):
    img_path = IMAGES_DIR + img_name
    feature = extract_features(img_path)
    features[img_name] = feature

print(f'Extracted features for {len(features)} images.')

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


  0%|          | 0/8091 [00:00<?, ?it/s]

Extracted features for 8091 images.
